# 농업 문서 요약 및 질의응답 — 노트북 예제

이 노트북은 `app.py`(Streamlit 앱)와 **같은 RAG 파이프라인**을 셀 단위로 실행합니다.

핵심 함수는 `app.py`에서 그대로 가져오므로, 웹 UI와 검색·답변 결과가 같습니다.

```text
PDF 로드 → 청킹 → OpenAI Embeddings → FAISS 저장
    → 질문 임베딩 → Top-K 검색 → LLM 생성
```

실행 전 `C:\env\.env`에 `OPENAI_API_KEY`가 있어야 합니다. 위에서부터 순서대로 실행하세요.


## 0. 라이브러리와 설정

`app.py`를 import하면 API 키 로드, 모델 이름, 청크 크기(800/150)가 함께 준비됩니다.


In [ ]:
from app import (
    BASE_DIR,
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    INDEX_DIR,
    build_faiss_index,
    compare_crops,
    embed_texts,
    get_client,
    index_signature,
    list_pdfs,
    load_and_chunk,
    load_saved_index,
    rag_answer,
    save_index,
    search,
    summarize_hits,
)

# 실습용 설정 (앱 사이드바와 같은 역할)
MAX_PAGES = 30
TOP_K = 4
REBUILD_INDEX = False  # True로 바꾸면 PDF를 다시 임베딩합니다.

print("작업 폴더:", BASE_DIR)
print("청크 크기 / 겹침:", CHUNK_SIZE, "/", CHUNK_OVERLAP)
print("인덱스 폴더:", INDEX_DIR)


## 1. PDF 목록 확인

같은 폴더의 농업기술길잡이 PDF를 찾습니다. 과제 조건상 **2개 이상**을 인덱싱합니다.


In [ ]:
pdfs = list_pdfs()
for i, p in enumerate(pdfs, start=1):
    print(f"{i:2d}. {p.name}")

SELECTED = [
    BASE_DIR / "농업기술길잡이40_딸기.PDF",
    BASE_DIR / "농업기술길잡이28_고구마.PDF",
]
SELECTED = [p for p in SELECTED if p.exists()]
if len(SELECTED) < 2:
    SELECTED = pdfs[:2]

print("\n인덱싱 대상:")
for p in SELECTED:
    print(" -", p.name)


## 2. PDF 로드 및 청킹

페이지별 텍스트를 추출한 뒤 `chunk_size=800`, `overlap=150`으로 자릅니다.

각 청크는 과제에서 요구한 Document 형태입니다.

- `page_content`: 본문
- `metadata.source`: 파일명
- `metadata.page`: 페이지 번호 (1부터)
- `metadata.chunk_id`: **같은 페이지 안에서의 조각 순서** (0, 1, 2, …)


In [ ]:
documents = load_and_chunk(SELECTED, max_pages=MAX_PAGES)
print(f"청크 수: {len(documents)}")
print("파일별 청크 수:")
from collections import Counter
print(Counter(d["metadata"]["source"] for d in documents))

print("\n--- 샘플 청크 1개 ---")
sample = documents[0]
print("source   :", sample["metadata"]["source"])
print("page     :", sample["metadata"]["page"])
print("chunk_id :", sample["metadata"]["chunk_id"])
print("본문 앞 300자:\n")
print(sample["page_content"][:300])


## 3. 임베딩 + FAISS 인덱스

청크 텍스트를 `text-embedding-3-small` 벡터로 바꾼 뒤 FAISS `IndexFlatIP`에 넣습니다.

벡터는 L2 정규화하므로, 검색 점수는 **코사인 유사도**입니다.

이미 `faiss_index/`가 있고 `REBUILD_INDEX=False`이면 저장된 인덱스를 재사용합니다. (앱의 영속화와 동일)


In [ ]:
saved = None if REBUILD_INDEX else load_saved_index()

if saved is not None:
    index, documents, signature = saved
    print("저장된 인덱스를 로드했습니다.")
    print("청크 수:", len(documents))
    print("설정:", signature)
else:
    print("임베딩을 시작합니다. 청크가 많으면 1~2분 걸릴 수 있습니다.")
    client = get_client()
    vectors = embed_texts(client, [d["page_content"] for d in documents])
    index = build_faiss_index(vectors)
    signature = index_signature([p.name for p in SELECTED], MAX_PAGES)
    save_index(index, documents, signature)
    print("인덱스 저장 완료")
    print("벡터 shape:", vectors.shape)
    print("FAISS ntotal:", index.ntotal)

client = get_client()
print("OpenAI 클라이언트 준비 완료")


## 4. 유사도 검색 확인

질문을 같은 모델로 임베딩한 뒤, FAISS에서 Top-K 청크를 가져옵니다.

키워드가 일치하는지가 아니라 **의미가 가까운지**를 봅니다.


In [ ]:
def show_hits(hits):
    for i, hit in enumerate(hits, start=1):
        meta = hit["metadata"]
        score = hit.get("score")
        score_txt = f"{score:.3f}" if isinstance(score, float) else "-"
        preview = hit["page_content"].replace("\n", " ")[:120]
        print(f"{i}) {meta['source']} / p.{meta['page']} / chunk_id={meta['chunk_id']} / 유사도 {score_txt}")
        print(f"   {preview}...")
        print()

query = "딸기 정식 시기와 주의점은?"
hits = search(client, index, documents, query, TOP_K)
print("질문:", query)
print()
show_hits(hits)


## 5. RAG 질의응답

검색된 청크만 프롬프트에 넣고 `gpt-4o-mini`가 답합니다.

컨텍스트에 없으면 `문서에서 확인되지 않습니다`라고 답하도록 시스템 프롬프트가 고정되어 있습니다.


In [ ]:
test_questions = [
    "딸기 정식 시기와 주의점은?",
    "고구마 저장 방법은?",
    "우주선은 어떻게 만드나요?",  # 문서에 없는 질문
]

for q in test_questions:
    print("=" * 60)
    print("질문:", q)
    hits = search(client, index, documents, q, TOP_K)
    answer = rag_answer(client, q, hits)
    print("\n답변:\n", answer)
    print("\n근거:")
    show_hits(hits)


### 직접 질문해 보기

아래 `my_question`만 바꿔 실행하면 됩니다.


In [ ]:
my_question = "고구마 저장 방법은?"

hits = search(client, index, documents, my_question, TOP_K)
answer = rag_answer(client, my_question, hits)

print("질문:", my_question)
print("\n답변:\n", answer)
print("\n근거:")
show_hits(hits)


## 6. 문서 요약

앱의 요약 탭과 같습니다.

- **주제 요약**: 주제로 검색한 청크만 요약
- **선택 문서 요약**: 해당 PDF 앞부분 청크를 모아 요약


In [ ]:
topic = "딸기 정식 시기와 주의사항"
topic_hits = search(client, index, documents, topic, TOP_K)
topic_summary = summarize_hits(client, topic, topic_hits)

print("[주제 요약]", topic)
print(topic_summary)
print("\n근거:")
show_hits(topic_hits)


In [ ]:
doc_name = SELECTED[0].name
sample = [
    {
        "page_content": d["page_content"],
        "metadata": d["metadata"],
        "score": None,
    }
    for d in documents
    if d["metadata"]["source"] == doc_name
][:12]

doc_summary = summarize_hits(client, f"{doc_name} 핵심 내용", sample)
print("[선택 문서 요약]", doc_name)
print(doc_summary)


## 7. 작물 비교 요약 (선택 과제)

두 문서와 항목(정식 조건, 병해충, 저장 등)을 넣어 비교합니다.


In [ ]:
crop_a = SELECTED[0].name
crop_b = SELECTED[1].name
topics = ["정식 조건", "병해충", "저장"]

hits_by_topic = {}
all_hits = []
for topic in topics:
    q = f"{crop_a}와 {crop_b}의 {topic}"
    h = search(client, index, documents, q, TOP_K)
    hits_by_topic[topic] = h
    all_hits.extend(h)

comparison = compare_crops(client, crop_a, crop_b, topics, hits_by_topic)
print(f"{crop_a}  vs  {crop_b}")
print()
print(comparison)


## 8. 정리

| 단계 | 함수 | 앱에서 대응하는 위치 |
|------|------|----------------------|
| PDF 목록 | `list_pdfs` | 사이드바 문서 선택 |
| 청킹 | `load_and_chunk` | 인덱스 생성 |
| 임베딩·FAISS | `embed_texts`, `build_faiss_index` | 인덱스 생성 |
| 저장/로드 | `save_index`, `load_saved_index` | 저장된 인덱스 불러오기 |
| 검색 | `search` | 모든 탭의 근거 청크 |
| 질의응답 | `rag_answer` | 탭: 질의응답 |
| 요약 | `summarize_hits` | 탭: 문서 요약 |
| 비교 | `compare_crops` | 탭: 작물 비교 요약 |

웹 화면으로 쓰려면 터미널에서 `streamlit run app.py`를 실행하면 됩니다.
